In [1]:
from pathlib import Path
import json
import re
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

warnings.filterwarnings("ignore")

try:
    display
except NameError:
    def display(obj):
        print(obj)

## 1. Duong dan va cau hinh

In [ ]:
def find_project_root(start_path):
    start_path = Path(start_path).resolve()
    candidate_paths = [start_path] + list(start_path.parents)
    for candidate in candidate_paths:
        data_file = candidate / "data" / "processed" / "student_voice_enriched_reviewed.csv"
        if data_file.exists():
            return candidate


PROJECT_ROOT = find_project_root(Path.cwd())
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "student_voice_enriched_reviewed.csv"
REPORT_DIR = PROJECT_ROOT / "outputs" / "reports"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
MODEL_DIR = PROJECT_ROOT / "outputs" / "models" / "baseline"

REPORT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = REPORT_DIR / "baseline_results.csv"
BASELINE_REPORT = REPORT_DIR / "baseline_report.md"

RANDOM_STATE = 42

print("Data:", DATA_PATH)
print("Results:", RESULTS_CSV)
print("Report:", BASELINE_REPORT)
print("Figures:", FIGURE_DIR)
print("Models:", MODEL_DIR)

Data: D:\AI thuc chien\Student Voice Intelligence\data\processed\student_voice_enriched_reviewed.csv
Results: D:\AI thuc chien\Student Voice Intelligence\outputs\reports\baseline_results.csv
Report: D:\AI thuc chien\Student Voice Intelligence\outputs\reports\baseline_report.md
Figures: D:\AI thuc chien\Student Voice Intelligence\outputs\figures


## 2. Load data va validate schema

In [ ]:
df = pd.read_csv(DATA_PATH, low_memory=False)
df["text"] = df["text"].fillna("").astype(str)

REQUIRED_COLUMNS = [
    "text",
    "split_original",
    "source_dataset",
    "sentiment_std_3class",
    "topic_group",
    "topic_std",
    "is_toxic",
    "urgency_level_final",
]

missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Data thieu cot bat buoc: {missing_columns}")

print("Shape:", df.shape)
display(df[REQUIRED_COLUMNS].head())

print("\nSplit distribution:")
display(df["split_original"].value_counts().to_frame("count"))

Shape: (49141, 25)


,text,split_original,source_dataset,sentiment_std_3class,topic_group,topic_std,is_toxic,urgency_level_final
0,em xin chào các anh chị em em là sinh viên vừa...,train,NEU_ESC,neutral,teaching_learning,academic,0,low
1,xây dựng mô hình sư phạm chuẩn mực sáng tạo ti...,train,NEU_ESC,neutral,teaching_learning,academic,0,low
2,sao lại ghét cái kiểu làm sai xong khóc lóc ăn...,train,NEU_ESC,negative,others,others,0,low
3,chào thầy đăng ký học ghép . môn học hiện hệ t...,train,NEU_ESC,neutral,teaching_learning,academic,0,low
4,con bé vẫn hoang mang lắm .,train,NEU_ESC,negative,others,others,0,low



Split distribution:


,count
split_original,
train,34474
test,9779
validation,4888


## 3. Cau hinh task


In [4]:
TASKS = [
    {
        "task_name": "sentiment_3class",
        "target_col": "sentiment_std_3class",
        "enabled": True,
        "class_weight": None,
        "notes": "Task chinh, label sach nhat.",
    },
    {
        "task_name": "topic_group",
        "target_col": "topic_group",
        "enabled": True,
        "class_weight": "balanced",
        "notes": "Topic taxonomy chung da gom nhom, van lech lop nen dung macro-F1.",
    },
    {
        "task_name": "toxic_binary",
        "target_col": "is_toxic",
        "enabled": True,
        "class_weight": "balanced",
        "notes": "Task mo rong, label seed+rule, lech lop manh.",
    },
    {
        "task_name": "urgency_final",
        "target_col": "urgency_level_final",
        "enabled": True,
        "class_weight": "balanced",
        "notes": "Task mo rong, LLM-reviewed candidates, high rat it.",
    },
]

task_summary = []
for task in TASKS:
    if not task["enabled"]:
        continue
    target = task["target_col"]
    task_summary.append({
        "task_name": task["task_name"],
        "target_col": target,
        "n_classes": df[target].nunique(dropna=False),
        "missing": int(df[target].isna().sum()),
        "notes": task["notes"],
    })

display(pd.DataFrame(task_summary))

,task_name,target_col,n_classes,missing,notes
0,sentiment_3class,sentiment_std_3class,3,0,"Task chinh, label sach nhat."
1,topic_group,topic_group,8,0,"Topic taxonomy chung da gom nhom, van lech lop..."
2,toxic_binary,is_toxic,2,0,"Task mo rong, label seed+rule, lech lop manh."
3,urgency_final,urgency_level_final,3,0,"Task mo rong, LLM-reviewed candidates, high ra..."


## 4. Ham tien xu ly cho baseline


In [5]:
def normalize_text_for_baseline(text):
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


df["text_for_model"] = df["text"].apply(normalize_text_for_baseline)

print(df["text_for_model"].head().to_string(index=False))

em xin chào các anh chị em em là sinh viên vừa ...
xây dựng mô hình sư phạm chuẩn mực sáng tạo tiê...
sao lại ghét cái kiểu làm sai xong khóc lóc ăn ...
chào thầy đăng ký học ghép . môn học hiện hệ th...
                       con bé vẫn hoang mang lắm .


## 5. Xu ly duplicate conflict theo tung target


In [6]:
def remove_train_label_conflicts(data, target_col):
    train_df = data[data["split_original"] == "train"].copy()
    conflict_texts = (
        train_df.groupby("text_for_model")[target_col]
        .nunique(dropna=False)
        .loc[lambda s: s > 1]
        .index
    )

    if len(conflict_texts) == 0:
        return data.copy(), pd.DataFrame()

    conflict_rows = data[(data["split_original"] == "train") & (data["text_for_model"].isin(conflict_texts))].copy()
    cleaned = data.drop(index=conflict_rows.index).copy()
    return cleaned, conflict_rows


for task in TASKS:
    if not task["enabled"]:
        continue
    _, conflicts = remove_train_label_conflicts(df, task["target_col"])
    print(task["task_name"], "conflict rows dropped from train:", len(conflicts))
    if len(conflicts) > 0:
        display(conflicts[["text", "split_original", task["target_col"]]].head())

sentiment_3class conflict rows dropped from train: 2


,text,split_original,sentiment_std_3class
44259,"thầy dạy hay , tuy nhiên còn nhiều chỗ chưa th...",train,positive
44383,"thầy dạy hay , tuy nhiên còn nhiều chỗ chưa th...",train,negative


topic_group conflict rows dropped from train: 0
toxic_binary conflict rows dropped from train: 0
urgency_final conflict rows dropped from train: 0


## 6. Ham train/evaluate model


In [7]:
def make_tfidf_vectorizer():
    return TfidfVectorizer(
        analyzer="word",
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        max_features=120_000,
        sublinear_tf=True,
    )


def make_models(class_weight=None):
    return {
        "tfidf_logreg": Pipeline([
            ("tfidf", make_tfidf_vectorizer()),
            ("clf", LogisticRegression(
                max_iter=1000,
                class_weight=class_weight,
                solver="saga",
                n_jobs=-1,
                random_state=RANDOM_STATE,
            )),
        ]),
        "tfidf_linear_svm": Pipeline([
            ("tfidf", make_tfidf_vectorizer()),
            ("clf", LinearSVC(
                class_weight=class_weight,
                random_state=RANDOM_STATE,
            )),
        ]),
    }


def compute_metrics(y_true, y_pred):
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "macro_f1": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "weighted_f1": f1_weighted,
    }


def save_confusion_matrix(y_true, y_pred, labels, title, filename):
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    plt.figure(figsize=(max(6, len(labels) * 0.8), max(5, len(labels) * 0.7)))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.colorbar(label="count")
    tick_marks = np.arange(len(labels))
    plt.xticks(tick_marks, labels, rotation=45, ha="right")
    plt.yticks(tick_marks, labels)
    plt.xlabel("Predicted")
    plt.ylabel("True")

    threshold = cm.max() / 2 if cm.size else 0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=8)

    plt.tight_layout()
    output_path = FIGURE_DIR / filename
    plt.savefig(output_path, dpi=160)
    plt.close()
    return output_path

## 7. Chay baseline cho tung task


In [ ]:
all_results = []
all_reports = {}
confusion_paths = []
model_paths = []

for task in TASKS:
    if not task["enabled"]:
        continue

    task_name = task["task_name"]
    target_col = task["target_col"]
    class_weight = task["class_weight"]

    print("Task:", task_name, "| target:", target_col)

    task_df = df[["text_for_model", "split_original", target_col]].dropna().copy()
    task_df, conflicts = remove_train_label_conflicts(task_df, target_col)

    train_df = task_df[task_df["split_original"] == "train"].copy()
    val_df = task_df[task_df["split_original"] == "validation"].copy()
    test_df = task_df[task_df["split_original"] == "test"].copy()

    labels = sorted(task_df[target_col].dropna().unique().tolist(), key=lambda x: str(x))

    print("Train/Val/Test:", len(train_df), len(val_df), len(test_df))
    print("Labels:", labels)
    print("Train distribution:")
    display(train_df[target_col].value_counts().to_frame("count"))

    models = make_models(class_weight=class_weight)

    for model_name, model in models.items():
        print("Training", model_name)
        model.fit(train_df["text_for_model"], train_df[target_col])

        model_path = MODEL_DIR / f"{task_name}_{model_name}.joblib"
        joblib.dump(model, model_path)
        model_paths.append(model_path)
        print("Saved model:", model_path)

        for split_name, eval_df in [("validation", val_df), ("test", test_df)]:
            y_true = eval_df[target_col]
            y_pred = model.predict(eval_df["text_for_model"])
            metrics = compute_metrics(y_true, y_pred)

            row = {
                "task_name": task_name,
                "target_col": target_col,
                "model_name": model_name,
                "split": split_name,
                "train_rows": len(train_df),
                "eval_rows": len(eval_df),
                "n_classes": len(labels),
                "class_weight": str(class_weight),
                "train_conflict_rows_dropped": len(conflicts),
                **metrics,
            }
            all_results.append(row)

            report_text = classification_report(y_true, y_pred, labels=labels, zero_division=0)
            all_reports[(task_name, model_name, split_name)] = report_text

            if split_name == "test":
                fig_name = f"baseline_confusion_matrix_{task_name}_{model_name}.png"
                fig_path = save_confusion_matrix(
                    y_true,
                    y_pred,
                    labels=labels,
                    title=f"{task_name} - {model_name} - test",
                    filename=fig_name,
                )
                confusion_paths.append(fig_path)

results_df = pd.DataFrame(all_results)
display(results_df.sort_values(["task_name", "split", "macro_f1"], ascending=[True, True, False]))

Task: sentiment_3class | target: sentiment_std_3class
Train/Val/Test: 34472 4888 9779
Labels: ['negative', 'neutral', 'positive']
Train distribution:


,count
sentiment_std_3class,
neutral,16394
negative,9539
positive,8539


Training tfidf_logreg
Training tfidf_linear_svm
Task: topic_group | target: topic_group
Train/Val/Test: 34474 4888 9779
Labels: ['career_jobs', 'events_news', 'facilities', 'others', 'personal_social', 'spam', 'student_services', 'teaching_learning']
Train distribution:


,count
topic_group,
teaching_learning,17722
others,10640
student_services,2112
personal_social,1568
events_news,1090
career_jobs,564
facilities,497
spam,281


Training tfidf_logreg
Training tfidf_linear_svm
Task: toxic_binary | target: is_toxic
Train/Val/Test: 34474 4888 9779
Labels: [0, 1]
Train distribution:


,count
is_toxic,
0,33688
1,786


Training tfidf_logreg
Training tfidf_linear_svm
Task: urgency_final | target: urgency_level_final
Train/Val/Test: 34474 4888 9779
Labels: ['high', 'low', 'medium']
Train distribution:


,count
urgency_level_final,
low,34207
medium,233
high,34


Training tfidf_logreg
Training tfidf_linear_svm


,task_name,target_col,model_name,split,train_rows,eval_rows,n_classes,class_weight,train_conflict_rows_dropped,accuracy,precision_macro,recall_macro,macro_f1,precision_weighted,recall_weighted,weighted_f1
3,sentiment_3class,sentiment_std_3class,tfidf_linear_svm,test,34472,9779,3,None,2,0.819409,0.822789,0.803536,0.811936,0.820368,0.819409,0.818725
1,sentiment_3class,sentiment_std_3class,tfidf_logreg,test,34472,9779,3,None,2,0.815421,0.828076,0.789948,0.804914,0.819593,0.815421,0.813599
2,sentiment_3class,sentiment_std_3class,tfidf_linear_svm,validation,34472,4888,3,None,2,0.833674,0.837535,0.818844,0.827097,0.834357,0.833674,0.832919
0,sentiment_3class,sentiment_std_3class,tfidf_logreg,validation,34472,4888,3,None,2,0.833061,0.847305,0.808152,0.823596,0.837527,0.833061,0.831230
7,topic_group,topic_group,tfidf_linear_svm,test,34474,9779,8,balanced,0,0.815421,0.658953,0.658603,0.658308,0.816581,0.815421,0.815736
5,topic_group,topic_group,tfidf_logreg,test,34474,9779,8,balanced,0,0.686266,0.523575,0.704522,0.567046,0.790968,0.686266,0.715777
6,topic_group,topic_group,tfidf_linear_svm,validation,34474,4888,8,balanced,0,0.820376,0.684707,0.694904,0.689106,0.820252,0.820376,0.820116
4,topic_group,topic_group,tfidf_logreg,validation,34474,4888,8,balanced,0,0.688625,0.514827,0.730929,0.564252,0.793901,0.688625,0.716685
11,toxic_binary,is_toxic,tfidf_linear_svm,test,34474,9779,2,balanced,0,0.991717,0.943910,0.866344,0.901225,0.991300,0.991717,0.991321
9,toxic_binary,is_toxic,tfidf_logreg,test,34474,9779,2,balanced,0,0.754985,0.541971,0.857569,0.506510,0.977391,0.754985,0.840209


## 8. Luu ket qua baseline

In [9]:
results_df.to_csv(RESULTS_CSV, index=False, encoding="utf-8")

best_test = (
    results_df[results_df["split"] == "test"]
    .sort_values(["task_name", "macro_f1"], ascending=[True, False])
    .groupby("task_name")
    .head(1)
    .reset_index(drop=True)
)

display(best_test[["task_name", "model_name", "accuracy", "macro_f1", "weighted_f1"]])

print("Saved:", RESULTS_CSV)

,task_name,model_name,accuracy,macro_f1,weighted_f1
0,sentiment_3class,tfidf_linear_svm,0.819409,0.811936,0.818725
1,topic_group,tfidf_linear_svm,0.815421,0.658308,0.815736
2,toxic_binary,tfidf_linear_svm,0.991717,0.901225,0.991321
3,urgency_final,tfidf_linear_svm,0.996421,0.751368,0.996136


Saved: D:\AI thuc chien\Student Voice Intelligence\outputs\reports\baseline_results.csv


## 9. Ghi markdown report

In [10]:
def md_table(table):
    try:
        return table.to_markdown(index=False)
    except ImportError:
        return "```text\n" + table.to_string(index=False) + "\n```"


report_lines = [
    "# Baseline Model Report",
    "",
    "## Input",
    "",
    f"- Data: `{DATA_PATH.relative_to(PROJECT_ROOT)}`",
    f"- Rows: `{len(df)}`",
    "",
    "## Models",
    "",
    "- TF-IDF word unigram+bigram + Logistic Regression",
    "- TF-IDF word unigram+bigram + Linear SVM",
    "",
    "## Best Test Results By Task",
    "",
    md_table(best_test[["task_name", "model_name", "accuracy", "macro_f1", "weighted_f1", "class_weight"]]),
    "",
    "## All Results",
    "",
    md_table(results_df[[
        "task_name", "model_name", "split", "accuracy", "precision_macro", "recall_macro", "macro_f1", "weighted_f1"
    ]].sort_values(["task_name", "split", "model_name"])),
    "",
    "## Confusion Matrices",
    "",
]

for path in confusion_paths:
    report_lines.append(f"- `{path.relative_to(PROJECT_ROOT)}`")

report_lines.extend([
    "",
    "## Saved Models",
    "",
])

for path in model_paths:
    report_lines.append(f"- `{path.relative_to(PROJECT_ROOT)}`")

report_lines.extend([
    "",
    "## Classification Reports",
    "",
])

for key, report_text in all_reports.items():
    task_name, model_name, split_name = key
    report_lines.extend([
        f"### {task_name} | {model_name} | {split_name}",
        "",
        "```text",
        report_text,
        "```",
        "",
    ])

report_lines.extend([
    "## Notes",
    "",
    "- Sentiment la task sach nhat va nen la baseline chinh.",
    "- Topic dung `topic_group` de train chung NEU+UIT; neu dung `topic_std`, nen train rieng theo source.",
    "- Toxic va urgency lech lop manh, nen doc macro-F1/recall tung lop thay vi accuracy.",
    "- Duplicate conflict trong train duoc drop theo tung target truoc khi train.",
])

BASELINE_REPORT.write_text("\n".join(report_lines), encoding="utf-8")

print("Saved report:", BASELINE_REPORT)

Saved report: D:\AI thuc chien\Student Voice Intelligence\outputs\reports\baseline_report.md
